# 2. Data Cleaning


In [25]:
import matplotlib.pyplot as plt # visualization
import numpy as np # numerical calcu
import pandas as pd # table operations
import seaborn as sns # rich visualization with mathplit
import warnings # avoid warnings
from scipy.stats import norm # statistical data
warnings.filterwarnings("ignore")

Input: `usedcars_stage1_understood.csv` from Data Understanding.

In [26]:
df = pd.read_csv('usedcars_stage1_understood.csv')
df.head()

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


In [27]:
df.drop(columns='Unnamed: 0',inplace=True)

Deleting Unnamed: 0 column which is the ID for the rows and is not meaningfull in the analysis

In [28]:
keyCol = ['Fuel_Type','Owner_Type'];
for col in keyCol:
  print(f"{col} has {df[col].unique()} values")

Fuel_Type has ['Diesel' 'Petrol' 'Electric'] values
Owner_Type has ['First' 'Second' 'Fourth & Above' 'Third'] values


Inference:
1) Fuel_Type includes Diesel,Petrol,Electric (with proper case sensitivly)
2) Owner_Type includes First,Second,Thrid, Fourth & Above (with proper case sensitivly)

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5847 entries, 0 to 5846
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               5847 non-null   object 
 1   Location           5847 non-null   object 
 2   Year               5847 non-null   int64  
 3   Kilometers_Driven  5847 non-null   int64  
 4   Fuel_Type          5847 non-null   object 
 5   Transmission       5847 non-null   object 
 6   Owner_Type         5847 non-null   object 
 7   Mileage            5845 non-null   object 
 8   Engine             5811 non-null   object 
 9   Power              5811 non-null   object 
 10  Seats              5809 non-null   float64
 11  New_Price          815 non-null    object 
 12  Price              5847 non-null   float64
dtypes: float64(2), int64(2), object(9)
memory usage: 594.0+ KB


In [30]:
df[(df['Seats'].notna()) & (df['Seats'] % 1 != 0)]

,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price


Inference : Seats have only integer values and hehce can be converted from float64 to int64

In [31]:
df['Name'] = df['Name'].astype('string')
df['Location'] = df['Location'].astype('string')
df['Location'] = df['Location'].astype('string')
df['Fuel_Type'] = df['Fuel_Type'].astype('string')
df['Fuel_Type'] = df['Fuel_Type'].astype('string')
df['Transmission'] = df['Transmission'].astype('string')
df['Owner_Type'] = df['Owner_Type'].astype('string')
df['Mileage'] = df['Mileage'].astype('string')
df['Engine'] = df['Engine'].astype('string')
df['New_Price'] = df['New_Price'].astype('string')
df['Seats'] = df['Seats'].astype('Int64')
df['Power'] = df['Power'].astype('string')

Inference: Changing the type of the object to string for easier analysis

In [32]:
df = df.rename(columns={'Year': 'Manufacturing_Year', 'New_Price': 'Showroom_Price'})

Inference:Renaming Year and New_Price to Manufacturing_Year and Showroom_Price respectively to avoid confusions in the name

In [33]:
df.duplicated().sum()

np.int64(0)

Inference:No duplicates found in the dataset

In [34]:
df.isna().sum()

,0
Name,0
Location,0
Manufacturing_Year,0
Kilometers_Driven,0
Fuel_Type,0
Transmission,0
Owner_Type,0
Mileage,2
Engine,36
Power,36


Inference : Found 5032 out of 5847 null values in "Showroom_Price" column and cannot generate much meaningfull data from it

In [35]:
df.drop(columns="Showroom_Price",inplace=True)

Droping Showroom_Price due to too many missing values

In [36]:
engineInCC    = df['Engine'].str.contains('CC', na=False).sum()
powerInBhp    = df['Power'].str.contains('bhp', na=False).sum()
mileageInkmpl = df['Mileage'].str.contains('kmpl', na=False).sum()
print(f"Engine column contains {engineInCC} rows in units CC")
print(f"Power column contains {powerInBhp} rows in units bph")
print(f"Mileage column contains {mileageInkmpl} rows in units kmpl")


Engine column contains 5811 rows in units CC
Power column contains 5811 rows in units bph
Mileage column contains 5842 rows in units kmpl


Inference : All notnull (5847-36 = 5811) rows in Engine,Power columns have the same units CC,bph respectively. Hence can be converted into float without any units conversion. But Mileage has 3 rows which are not in km/kg (5847-5832-2=3) and needs to be converted

In [37]:
df[(~df['Mileage'].str.contains('kmpl', na=False)) & (df['Mileage'].notna())]

,Name,Location,Manufacturing_Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price
1,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5,4.50
24,Nissan Micra Diesel XV,Hyderabad,2012,54000,Diesel,Manual,First,16.48 km/kg,1461 CC,63.1 bhp,5,4.25
56,Nissan X-Trail SLX AT,Hyderabad,2010,121812,Diesel,Automatic,First,9.49 km/kg,1995 CC,147.6 bhp,5,7.75


Inference: 3 rows have mileage in km/kg

In [38]:
density = {'Petrol': 0.74, 'Diesel': 0.84}
is_kmkg = df['Mileage'].str.contains('km/kg', na=False)
value = df['Mileage'].str.extract(r'([\d.]+)', expand=False).astype(float)

factor = df['Fuel_Type'].map(density)
df['Mileage'] = df['Mileage'].round(2).astype(str) + ' kmpl'

Inference:Converted the mileage to kmpl by using the relation <br>
kmpl = (km/kg) x density in kg per liter fuel

In [39]:
for col in ['Mileage', 'Engine', 'Power']:
    df[col] = df[col].astype(str).str.extract(r'([\d.]+)')[0].astype(float)

Inference: Converted the columns Mileage,Engine,Power from string to float

In [40]:
df.isna().sum()

,0
Name,0
Location,0
Manufacturing_Year,0
Kilometers_Driven,0
Fuel_Type,0
Transmission,0
Owner_Type,0
Mileage,2
Engine,36
Power,36


Inference : No rows had values like NaN units in Engine,Power,Mileage

In [41]:
df.describe()

,Manufacturing_Year,Kilometers_Driven,Mileage,Engine,Power,Seats,Price
count,5847.000000,5.847000e+03,5845.000000,5811.000000,5811.000000,5809.0,5847.000000
mean,2013.448435,5.841013e+04,18.158496,1631.552573,113.803144,5.286452,9.653742
std,3.194949,9.237971e+04,4.358246,601.972587,53.896719,0.806668,11.275966
min,1998.000000,1.710000e+02,0.000000,72.000000,34.200000,2.0,0.440000
25%,2012.000000,3.346750e+04,15.260000,1198.000000,78.000000,5.0,3.550000
50%,2014.000000,5.257600e+04,18.190000,1497.000000,98.600000,5.0,5.750000
75%,2016.000000,7.249050e+04,21.100000,1991.000000,139.010000,5.0,10.250000
max,2019.000000,6.500000e+06,28.400000,5998.000000,560.000000,10.0,160.000000


Inference:
1) The dataset has records of manufacturing year from 1998 to 2019
2) Kilometer Driver ranges from 171 to 65,00,000 ( 65,00,000 can be a high value )
3) Mileage ranges from 0 to 28.4 kmpl ( 0 may be electric, upper mileage limit is < 35 )
4) Power ranges from 34.2 to 560 bhp
5) Seats ranges from 2 to 10
6) Price ranges from 0.44 to 160 (lakhs not exactly mentioned)

In [42]:
df[df['Fuel_Type'] == 'Electric']

,Name,Location,Manufacturing_Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price
4313,Mahindra E Verito D4,Chennai,2016,50000,Electric,Automatic,First,NaN,72.0,41.0,5,13.00
4762,Toyota Prius 2009-2016 Z4,Mumbai,2011,44000,Electric,Automatic,First,NaN,1798.0,73.0,5,12.75


Inference:No electric cars in the above dataset have 0 mileage but have Engine in CC, which does not mean anything and is an error

In [43]:
df.loc[df['Fuel_Type'] == 'Electric', 'Engine'] = np.nan

In [44]:
print(f"{'Mileage'}: {(df['Mileage'] < 0).sum()} negative, {(df['Mileage'] == 0).sum()} zero")

Mileage: 0 negative, 40 zero


Inference:Mileage has 40 values that are zero,no other columns have impossible values

In [45]:
df[df['Mileage'] == 0]['Name'].value_counts()

,count
Name,
Mercedes-Benz M-Class ML 350 4Matic,8
Hyundai Santro Xing GL,7
Land Rover Freelander 2 TD4 SE,4
Hyundai Santro GLS II - Euro II,4
Mercedes-Benz C-Class Progressive C 220d,3
Honda City 1.5 GXI,3
Land Rover Range Rover 3.0 D,2
Skoda Superb 3.6 V6 FSI,2
Mercedes-Benz New C-Class Progressive C 200,1


Inference: Looking at how clustered the 0's are to a particular model for replacing these null values

In [46]:
df[df['Mileage'] == 0]['Fuel_Type'].value_counts()

,count
Fuel_Type,
Diesel,21
Petrol,19


Inference:Electric cars already have Mileage NaN since, mileage in kmpl does not make sence to electric cars

In [47]:
df['Mileage'] = df['Mileage'].replace(0, np.nan)

Inference:Replacing the impossbile mileage to Null

In [48]:
df['Mileage'] = df['Mileage'].fillna(df.groupby('Name')['Mileage'].transform('median'))

In [49]:
df['Mileage'] = df['Mileage'].fillna(df.groupby('Fuel_Type')['Mileage'].transform('median'))

In [50]:
df[df['Mileage'].isna()][['Name','Fuel_Type']]

,Name,Fuel_Type
4313,Mahindra E Verito D4,Electric
4762,Toyota Prius 2009-2016 Z4,Electric


Inference:
1) Filled the NaN mileage of vehicle with the median value of cars with same name
2) Filled the NaN mileage of vehile other which did not have other similar vechile with the median of the fuel type
3) Remaning vehicle with NaN mileage is of Fuel_Type Electric

In [51]:
age = (2019 - df['Manufacturing_Year']).clip(lower=1)
df['km_per_year'] = df['Kilometers_Driven'] / age
df['km_per_year'].describe()

,km_per_year
count,5.847000e+03
mean,1.284475e+04
std,4.327922e+04
min,1.050833e+02
25%,7.000000e+03
50%,1.023077e+04
75%,1.489650e+04
max,3.250000e+06


Inference: The km drived per year has a maximum value of 3250000, signifying heavy car useage

In [52]:
sus = df[df['km_per_year'] > 100000]
sus

,Name,Location,Manufacturing_Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price,km_per_year
326,Skoda Octavia Ambition Plus 2.0 TDI AT,Kolkata,2013,775000,Diesel,Automatic,First,19.30,1968.0,141.00,5,7.5,1.291667e+05
1805,Volkswagen Vento Diesel Highline,Chennai,2013,720000,Diesel,Manual,First,20.54,1598.0,103.60,5,5.9,1.200000e+05
2259,BMW X5 xDrive 30d M Sport,Chennai,2017,6500000,Diesel,Automatic,First,15.97,2993.0,258.00,5,65.0,3.250000e+06
2734,Volkswagen Jetta 2013-2015 2.0L TDI Highline AT,Chennai,2015,480000,Diesel,Automatic,First,16.96,1968.0,138.03,5,13.0,1.200000e+05
2997,Honda City i VTEC SV,Kolkata,2015,480000,Petrol,Manual,First,17.40,1497.0,117.30,5,5.0,1.200000e+05


Inference: Cars are not newly manufactured for such high km

In [53]:
sus[['Kilometers_Driven', 'Price']].corr()

,Kilometers_Driven,Price
Kilometers_Driven,1.00000,0.98929
Price,0.98929,1.00000


Inference: The car has a positive correlation between Kilometers Driven and Price which should not be possible due to physical limits

In [54]:
sus.drop(sus['Kilometers_Driven'].idxmax())[['Kilometers_Driven', 'Price']].corr()

,Kilometers_Driven,Price
Kilometers_Driven,1.000000,-0.340279
Price,-0.340279,1.000000


Inference: One row with kilometer driven = 65,00,000 is driving the postive correlation

In [55]:
idx = sus['Kilometers_Driven'].idxmax()

In [56]:
df.loc[idx, 'Kilometers_Driven'] = df[(df['Name'] == df.loc[idx,"Name"]) & (df.index != idx)][['Kilometers_Driven']].median()[0]

Inference:Set the huge kilometer_driven to the median of kilometers_Driven of cars with same name (except itself) since it may be a error

In [57]:
df['bhp_per_cc'] = df['Power'] / df['Engine']
df['bhp_per_cc'].describe()


,bhp_per_cc
count,5809.000000
mean,0.068806
std,0.012928
min,0.023842
25%,0.061635
50%,0.067174
75%,0.075446
max,0.153421


Inference:Some cars have bhp_per_cc slighly greater than 0.13

In [58]:
df[df['bhp_per_cc'] > 0.15]

,Name,Location,Manufacturing_Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price,km_per_year,bhp_per_cc
140,Audi RS5 Coupe,Mumbai,2013,23000,Petrol,Automatic,First,11.05,2894.0,444.0,4,37.0,3833.333333,0.153421
3935,Audi RS5 Coupe,Mumbai,2013,23312,Petrol,Automatic,First,11.05,2894.0,444.0,4,40.5,3885.333333,0.153421


Inference: Audi RS5 Coupe has a v8 engine for high performance explaining the high result

In [59]:
q1, q3 = df['Price'].quantile([0.25, 0.75])
iqr = q3 - q1
df[df['Price'] > q3 + 1.5 * iqr]

,Name,Location,Manufacturing_Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price,km_per_year,bhp_per_cc
11,Land Rover Range Rover 2.2L Pure,Delhi,2014,72000,Diesel,Automatic,First,12.70,2179.0,187.70,5,27.00,14400.000000,0.086140
17,Mercedes-Benz New C-Class C 220 CDI BE Avantgare,Bangalore,2014,78500,Diesel,Automatic,First,14.84,2143.0,167.62,5,28.00,15700.000000,0.078217
20,Audi A6 2011-2015 35 TFSI Technology,Mumbai,2015,55985,Petrol,Automatic,First,13.53,1984.0,177.01,5,23.50,13996.250000,0.089219
36,Mercedes-Benz M-Class ML 350 CDI,Pune,2013,85000,Diesel,Automatic,First,11.74,2987.0,254.80,5,28.00,14166.666667,0.085303
48,Audi A4 2.0 TDI 177 Bhp Premium Plus,Kochi,2015,13648,Diesel,Automatic,First,17.11,1968.0,174.33,5,21.43,3412.000000,0.088582
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5776,BMW 5 Series 2013-2017 530d M Sport,Bangalore,2016,16000,Diesel,Automatic,First,14.69,2993.0,258.00,5,48.00,5333.333333,0.086201
5780,Audi Q3 2.0 TDI,Bangalore,2014,65630,Diesel,Automatic,First,15.73,1968.0,174.33,5,21.75,13126.000000,0.088582
5800,Mercedes-Benz GLA Class 200 Sport,Kochi,2018,17773,Petrol,Automatic,First,13.70,1991.0,183.00,5,26.76,17773.000000,0.091914
5825,Jaguar XF 2.2 Litre Luxury,Kochi,2016,31150,Diesel,Automatic,First,16.36,2179.0,187.70,5,30.54,10383.333333,0.086140


Inference: 679 cars prices are statistical outliers

In [60]:
df = df.drop(columns=['bhp_per_cc', 'km_per_year'])

Droping the rows used for finding upper limit values

In [61]:
df.isna().sum()

,0
Name,0
Location,0
Manufacturing_Year,0
Kilometers_Driven,0
Fuel_Type,0
Transmission,0
Owner_Type,0
Mileage,2
Engine,38
Power,36


In [62]:
non_ev = df['Fuel_Type'] != 'Electric'
df.loc[non_ev, 'Engine'] = df.loc[non_ev, 'Engine'].fillna(df.groupby('Name')['Engine'].transform('median'))
df.loc[non_ev, 'Engine'] = df.loc[non_ev, 'Engine'].fillna(df.groupby('Fuel_Type')['Engine'].transform('median'))

Inference:
1) Filled the NaN engine of vehicle with the median value of cars with same name
2) Filled the NaN engine of vehile other which did not have other similar vechile with the median of the fuel type

In [63]:
df['Power'] = df['Power'].fillna(df.groupby('Name')['Power'].transform('median'))
df['Power'] = df['Power'].fillna(df.groupby('Fuel_Type')['Power'].transform('median'))

Inference:
1) Filled the NaN power of vehicle with the median value of cars with same name
2) Filled the NaN power of vehile other which did not have other similar vechile with the median of the fuel type

In [64]:
df['Seats'] = df['Seats'].fillna(
    df.groupby('Name')['Seats'].transform(lambda s: s.mode().iloc[0] if s.notna().any() else np.nan))
df['Seats'] = df['Seats'].fillna(df['Seats'].mode()[0])

Inference:
1) Filled the NaN seats of vehicle with the mode value of cars with same name
2) Filled the NaN engine of vehile other which did not have other similar vechile with the mode of the seats column

In [65]:
df[df['Name'] == df.loc[idx,"Name"]][['Kilometers_Driven']].median()[0]

np.float64(33000.0)

In [66]:
df.isna().sum()

,0
Name,0
Location,0
Manufacturing_Year,0
Kilometers_Driven,0
Fuel_Type,0
Transmission,0
Owner_Type,0
Mileage,2
Engine,2
Power,0


Inference: Engine and Mileage is Nan for the 2 electric cars

## Export for the next stage

Run this notebook after Data Understanding. EDA reads this cleaned CSV.

In [68]:
df.to_csv('usedcars_stage2_cleaned.csv', index=False)
print('Exported usedcars_stage2_cleaned.csv')

Exported usedcars_stage2_cleaned.csv
